# MixID Tier-2 Enrichment (Colab GPU)

Upload a mix file and run Demucs vocal-stem separation, Whisper lyrics transcription, and pitch-swept Chromaprint fingerprinting on the **vocals stem only**. The vocals-only fingerprint defeats crowd noise and stem-swap mashups that confuse Tier-1 fingerprinting on the full mix.

**Before you start:** `Runtime → Change runtime type → GPU` (T4 free tier is enough).

**Output:** `enriched_tracklist.json` — download and pass to `mixid` locally to merge with Tier-1 results.

## 1. Install dependencies (~3 minutes one-time)

In [ ]:
!pip install -q demucs openai-whisper pyacoustid librosa soundfile requests
!apt-get install -y -qq libchromaprint-tools >/dev/null

## 2. Upload your mix (or paste a Drive/HTTPS URL)

In [ ]:
from google.colab import files
uploaded = files.upload()
MIX_PATH = next(iter(uploaded.keys()))
print('Got:', MIX_PATH)

## 3. Demucs: separate vocal stem

`htdemucs_ft` is the fine-tuned 4-stem model. On a T4 GPU, a 2-hour mix separates in ~8 minutes.

In [ ]:
import os, subprocess, time
os.makedirs('separated', exist_ok=True)
t0 = time.time()
subprocess.run([
    'python', '-m', 'demucs', '-n', 'htdemucs_ft',
    '--two-stems=vocals', '-o', 'separated', MIX_PATH,
], check=True)
print(f'Demucs done in {time.time()-t0:.0f}s')

stem_dir = f'separated/htdemucs_ft/{os.path.splitext(os.path.basename(MIX_PATH))[0]}'
VOCALS_PATH = os.path.join(stem_dir, 'vocals.wav')
NO_VOCALS_PATH = os.path.join(stem_dir, 'no_vocals.wav')
print('vocals:', VOCALS_PATH)
print('instrumental:', NO_VOCALS_PATH)

## 4. Find segment boundaries (same hybrid logic as Tier-1)

novelty curve over MFCC+chroma + a forced 90s gap cap. Re-implemented inline so the notebook stays self-contained.

In [ ]:
import librosa, numpy as np

y, sr = librosa.load(MIX_PATH, sr=22050, mono=True)
duration_sec = len(y) / sr

# Agglomerative segmentation on MFCC + chroma
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
feat = np.vstack([mfcc, chroma])
k = max(8, int(duration_sec / 60))
bounds_frames = librosa.segment.agglomerative(feat, k=k)
novelty_secs = librosa.frames_to_time(bounds_frames, sr=sr).tolist()

# Enforce 90s max gap
MAX_GAP = 90.0
all_bounds = sorted(set(novelty_secs))
augmented = []
last = 0.0
for b in all_bounds:
    while b - last > MAX_GAP:
        last += MAX_GAP
        augmented.append(last)
    augmented.append(b)
    last = b
while duration_sec - last > MAX_GAP:
    last += MAX_GAP
    augmented.append(last)

starts = [0.0] + augmented
ends = augmented + [duration_sec]
segments = [(s, e) for s, e in zip(starts, ends) if e - s > 5.0]
print(f'{duration_sec:.0f}s mix → {len(segments)} segments')

## 5. Pitch-swept Chromaprint of vocals stem per segment

The vocals stem is much less corrupted by crowd noise than the full mix. Combined with the ±6% pitch sweep that defeats DJ beatmatching, this is the biggest Tier-2 lever.

In [ ]:
import soundfile as sf, tempfile, json

vocals, sr_v = librosa.load(VOCALS_PATH, sr=22050, mono=True)
PITCH_SWEEP = (-6, -4, -2, 0, 2, 4, 6)
SAMPLE_WIN = 12


def best_sample(seg_start, seg_end, audio, sr):
    seg = audio[int(seg_start * sr):int(seg_end * sr)]
    if len(seg) < SAMPLE_WIN * sr:
        return None
    win = int(SAMPLE_WIN * sr)
    hop = sr
    n = (len(seg) - win) // hop + 1
    rms = np.array([np.sqrt(np.mean(seg[i*hop:i*hop+win]**2)) for i in range(n)])
    best = int(np.argmax(rms))
    return seg[best*hop:best*hop+win], best * hop / sr


def fpcalc_b64(samples, sr):
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
        sf.write(f.name, samples, sr, subtype='PCM_16')
        r = subprocess.run(['fpcalc', '-json', '-length', '120', f.name],
                           capture_output=True, text=True, timeout=30)
    return json.loads(r.stdout)


segment_fingerprints = []
for i, (s, e) in enumerate(segments):
    picked = best_sample(s, e, vocals, sr_v)
    if picked is None:
        continue
    samples, offset_in_seg = picked
    variants = []
    for pct in PITCH_SWEEP:
        if pct == 0:
            shifted = samples
        else:
            shifted = librosa.effects.pitch_shift(samples, sr=sr_v, n_steps=pct/100*12).astype(np.float32)
        fp = fpcalc_b64(shifted, sr_v)
        variants.append({'pitch_pct': pct, 'duration': fp['duration'], 'fingerprint': fp['fingerprint']})
    segment_fingerprints.append({
        'segment_index': i,
        'start_sec': s,
        'end_sec': e,
        'sample_start_in_segment': offset_in_seg,
        'variants': variants,
    })
print(f'fingerprinted {len(segment_fingerprints)} segments × {len(PITCH_SWEEP)} pitch variants each')

## 6. AcoustID lookup per segment (best variant wins)

Get a free API key at https://acoustid.org/api-key and paste below.

In [ ]:
ACOUSTID_API_KEY = 'PASTE_YOUR_KEY_HERE'  # noqa: E501

import acoustid, time
matches = []
for seg in segment_fingerprints:
    best = None
    for variant in sorted(seg['variants'], key=lambda v: abs(v['pitch_pct'])):
        try:
            resp = acoustid.lookup(ACOUSTID_API_KEY, variant['fingerprint'], int(variant['duration']), meta='recordings')
        except acoustid.WebServiceError as e:
            print('AcoustID error:', e); continue
        for r in (resp.get('results') or []):
            recs = r.get('recordings') or []
            if not recs: continue
            rec = recs[0]
            score = float(r.get('score', 0.0))
            artist = ', '.join(a.get('name','') for a in (rec.get('artists') or []))
            title = rec.get('title', '')
            if best is None or score > best['score']:
                best = {'score': score, 'artist': artist, 'title': title,
                        'pitch_pct': variant['pitch_pct'], 'recording_id': rec.get('id','')}
        if best and best['score'] >= 0.85:
            break  # short-circuit on confident hit
        time.sleep(0.34)  # 3 req/sec rate limit
    matches.append({
        'segment_index': seg['segment_index'],
        'start_sec': seg['start_sec'],
        'end_sec': seg['end_sec'],
        'best': best,
    })
    print(f"seg {seg['segment_index']:02d} ({int(seg['start_sec']):4d}-{int(seg['end_sec']):4d}s): "
          + (f"{best['artist']} - {best['title']} ({best['score']:.2f}, pitch {best['pitch_pct']:+d}%)" if best else 'no match'))

## 7. Optional: Whisper-small lyrics transcription on vocal stem

Whisper-small on T4 GPU is ~real-time. Pair with Genius search for tracks that AcoustID still missed.

In [ ]:
RUN_WHISPER = False  # set True to enable
if RUN_WHISPER:
    import whisper
    model = whisper.load_model('small')
    for m in matches:
        if m['best'] and m['best']['score'] >= 0.85:
            continue  # already confidently matched
        seg = segment_fingerprints[next(i for i, s in enumerate(segment_fingerprints) if s['segment_index'] == m['segment_index'])]
        snippet = vocals[int(m['start_sec']*sr_v):int(m['end_sec']*sr_v)]
        snippet = librosa.resample(snippet, orig_sr=sr_v, target_sr=16000).astype(np.float32)
        snippet = snippet[:30*16000] if len(snippet) > 30*16000 else snippet
        snippet = np.pad(snippet, (0, max(0, 30*16000 - len(snippet))))
        mel = whisper.log_mel_spectrogram(snippet).to(model.device)
        opts = whisper.DecodingOptions(language='en', without_timestamps=True, fp16=True)
        result = whisper.decode(model, mel, opts)
        m['lyric_phrase'] = (result.text or '').strip()
        print(f"seg {m['segment_index']:02d} lyrics: {m['lyric_phrase'][:100]}")

## 8. Save and download enriched tracklist

Use the resulting JSON with `python -m mixid.enrich.merge` locally to overlay onto your Tier-1 tracklist.

In [ ]:
out = {
    'source_mix': MIX_PATH,
    'method': 'tier2:demucs_vocals + chromaprint pitch_sweep + acoustid',
    'matches': matches,
}
with open('enriched_tracklist.json', 'w', encoding='utf-8') as f:
    json.dump(out, f, indent=2, ensure_ascii=False)
from google.colab import files as colab_files
colab_files.download('enriched_tracklist.json')